# 01 - Setup Lakebase PostgreSQL Instance

This notebook creates a Lakebase PostgreSQL instance for the Personal Expense Tracker.

**Features:**
- Create PostgreSQL instance using Databricks SDK
- Configure instance for development
- Retrieve connection details
- Enable SSL connection
- Error handling and validation


## Install Required Libraries


In [ ]:
%pip install databricks-sdk psycopg2-binary sqlalchemy --quiet
dbutils.library.restartPython()


## Import Libraries


In [ ]:
from databricks.sdk import WorkspaceClient
import time
import json
import os

# Note: Lakebase imports may vary based on SDK version
# If Lakebase is not available, this notebook will show the concepts


## Configuration


In [ ]:
# Instance configuration
INSTANCE_NAME = "expense-tracker-lakebase"
DATABASE_NAME = "expense_tracker_db"
INSTANCE_TYPE = "SMALL"  # Small instance for development
STORAGE_SIZE_GB = 100  # 100GB storage

print(f"Instance Name: {INSTANCE_NAME}")
print(f"Database Name: {DATABASE_NAME}")
print(f"Instance Type: {INSTANCE_TYPE}")
print(f"Storage Size: {STORAGE_SIZE_GB}GB")


## Initialize Databricks Workspace Client


In [ ]:
try:
    # Initialize workspace client
    w = WorkspaceClient()
    print("✓ Successfully initialized Databricks Workspace Client")
    print(f"✓ Workspace URL: {w.config.host}")
except Exception as e:
    print(f"✗ Failed to initialize workspace client: {str(e)}")
    raise


## Check if Database Already Exists


In [ ]:
def check_database_exists(database_name):
    """Check if a Lakebase database already exists"""
    try:
        # Check if database API is available
        if not hasattr(w, 'database'):
            print("Note: Lakebase API not available in this workspace")
            return False, None
            
        databases = w.database.list()
        for db in databases:
            if db.name == database_name:
                return True, db
        return False, None
    except AttributeError:
        print("Note: Lakebase is not available in this Databricks workspace")
        print("This is expected if Lakebase is not enabled or in preview")
        return False, None
    except Exception as e:
        print(f"Warning: Could not list databases: {str(e)}")
        print("Continuing with demo - you can skip to notebook 02 if Lakebase is unavailable")
        return False, None

exists, existing_db = check_database_exists(DATABASE_NAME)
if exists:
    print(f"✓ Database '{DATABASE_NAME}' already exists")
    print(f"  Database ID: {existing_db.id}")
    print(f"  State: {existing_db.state}")
else:
    print(f"✓ Database '{DATABASE_NAME}' does not exist or Lakebase not available")
    print("  For this demo, you can proceed to notebook 02 to work directly with Unity Catalog")


## Create Lakebase PostgreSQL Database


In [ ]:
def create_lakebase_database():
    """Create a Lakebase PostgreSQL database"""
    try:
        # Check if database API is available
        if not hasattr(w, 'database'):
            print("⚠️  Lakebase API not available")
            print("   For this demo, skip to notebook 02 to use Unity Catalog directly")
            return None
            
        print(f"Creating Lakebase database '{DATABASE_NAME}'...")
        
        # Create database using SDK
        database = w.database.create(
            name=DATABASE_NAME,
            instance_type=INSTANCE_TYPE,
            description="Personal Expense Tracker Database"
        )
        
        print(f"✓ Database creation initiated")
        print(f"  Database ID: {database.id}")
        print(f"  Database Name: {database.name}")
        
        return database
        
    except AttributeError as e:
        print(f"⚠️  Lakebase not available: {str(e)}")
        print("   This is expected if Lakebase is in preview or not enabled")
        print("   📌 Skip to notebook 02 to continue with Unity Catalog")
        return None
    except Exception as e:
        print(f"⚠️  Could not create database: {str(e)}")
        print("   📌 Skip to notebook 02 to continue with Unity Catalog")
        return None

# Create database if it doesn't exist
database = None
if not exists:
    database = create_lakebase_database()
else:
    database = existing_db
    print("Using existing database")

# Check if we should continue
if database is None and not exists:
    print("\n" + "="*70)
    print("LAKEBASE NOT AVAILABLE")
    print("="*70)
    print("Lakebase (PostgreSQL) is not available in this workspace.")
    print("\nThis is normal if:")
    print("  • Lakebase is in preview/not yet released")
    print("  • Your workspace doesn't have Lakebase enabled")
    print("  • You're using Community Edition")
    print("\n✅ NEXT STEPS:")
    print("  1. Skip the rest of this notebook")
    print("  2. Go directly to notebook 02-create-schema.ipynb")
    print("  3. In notebook 02, create tables in Unity Catalog instead")
    print("  4. Continue with notebook 03 for the sync pipeline")
    print("="*70)
    dbutils.notebook.exit("Lakebase not available - proceed to notebook 02")


## Wait for Database to be Ready


In [ ]:
def wait_for_database_ready(database_id, timeout_minutes=30):
    """Wait for the database to be in READY state"""
    if database_id is None:
        print("⚠️  No database ID - skipping wait")
        return None
        
    print(f"\nWaiting for database to be ready (timeout: {timeout_minutes} minutes)...")
    
    start_time = time.time()
    timeout_seconds = timeout_minutes * 60
    
    while True:
        try:
            # Get database status
            current_db = w.database.get(id=database_id)
            state = str(current_db.state) if hasattr(current_db, 'state') else 'UNKNOWN'
            
            elapsed_time = int(time.time() - start_time)
            print(f"  [{elapsed_time}s] Current state: {state}")
            
            # Check for ready states (state names may vary)
            if 'ACTIVE' in state.upper() or 'READY' in state.upper() or 'RUNNING' in state.upper():
                print(f"\n✓ Database is READY! (took {elapsed_time}s)")
                return current_db
            
            # Check for failed state
            if 'FAILED' in state.upper() or 'ERROR' in state.upper():
                raise Exception(f"Database provisioning failed with state: {state}")
            
            # Check timeout
            if elapsed_time > timeout_seconds:
                raise TimeoutError(f"Database creation timed out after {timeout_minutes} minutes")
            
            # Wait before checking again
            time.sleep(30)
                
        except Exception as e:
            print(f"\n✗ Error while waiting: {str(e)}")
            raise

# Wait for database to be ready
ready_database = None
if database is not None:
    ready_database = wait_for_database_ready(database.id)
else:
    print("⚠️  Skipping database wait - Lakebase not available")


## Retrieve Connection Details


In [ ]:
def get_connection_details(database_id):
    """Retrieve connection details for the database"""
    if database_id is None:
        return None
        
    try:
        # Get database details
        db = w.database.get(id=database_id)
        
        # Get connection string (method name may vary)
        try:
            connection_string = w.database.get_connection_string(id=database_id)
        except:
            connection_string = "Connection string retrieval not available"
        
        details = {
            "database_id": str(db.id) if hasattr(db, 'id') else database_id,
            "database_name": str(db.name) if hasattr(db, 'name') else DATABASE_NAME,
            "state": str(db.state) if hasattr(db, 'state') else "UNKNOWN",
            "connection_string": connection_string,
            "instance_type": INSTANCE_TYPE,
            "ssl_enabled": True
        }
        
        return details
        
    except Exception as e:
        print(f"✗ Failed to retrieve connection details: {str(e)}")
        return None

# Get connection details
connection_details = None
if ready_database is not None:
    connection_details = get_connection_details(ready_database.id if hasattr(ready_database, 'id') else None)
    
    if connection_details:
        print("\n" + "="*60)
        print("CONNECTION DETAILS")
        print("="*60)
        for key, value in connection_details.items():
            if key != "connection_string":  # Don't print full connection string
                print(f"{key:20s}: {value}")
        print("="*60)
else:
    print("\n⚠️  No connection details available - Lakebase not provisioned")
    connection_details = {
        "database_id": "not_available",
        "database_name": DATABASE_NAME,
        "state": "NOT_CREATED",
        "instance_type": INSTANCE_TYPE,
        "ssl_enabled": True
    }


## Save Configuration


In [ ]:
# Save configuration as notebook widget for easy access
if connection_details:
    dbutils.widgets.text("database_id", str(connection_details["database_id"]), "Database ID")
    dbutils.widgets.text("database_name", str(connection_details["database_name"]), "Database Name")
    
    print(f"\n✓ Configuration saved as notebook widgets")
    print(f"✓ Database ID: {connection_details['database_id']}")
    print(f"✓ Database Name: {connection_details['database_name']}")
else:
    print("\n⚠️  Skipping configuration save - no database created")


## Summary


In [ ]:
print("\n" + "="*70)
if connection_details and connection_details['state'] not in ['NOT_CREATED', 'UNKNOWN']:
    print("✅ LAKEBASE SETUP COMPLETE")
    print("="*70)
    print(f"✓ Lakebase PostgreSQL database created: {DATABASE_NAME}")
    print(f"✓ Database ID: {connection_details['database_id']}")
    print(f"✓ State: {connection_details['state']}")
    print(f"✓ Instance Type: {INSTANCE_TYPE}")
    print(f"✓ SSL: Enabled")
    print(f"\n📌 Next Steps:")
    print(f"  1. Run notebook 02-create-schema.ipynb to create database schema")
    print(f"  2. Insert sample expense data")
    print(f"  3. Set up sync pipeline to Lakehouse")
else:
    print("⚠️  LAKEBASE NOT AVAILABLE - USING ALTERNATIVE PATH")
    print("="*70)
    print("Lakebase is not available in this workspace.")
    print("\n✅ This is OK! You can still complete the project using Unity Catalog.")
    print("\n📌 Alternative Path:")
    print("  1. SKIP this notebook's remaining steps")
    print("  2. Go to notebook 02-create-schema.ipynb")
    print("  3. In notebook 02, SQL cells will create tables directly")
    print("  4. Continue with notebook 03-sync-pipeline.ipynb")
    print("  5. Complete the project using Unity Catalog as both OLTP and OLAP")
    print("\n💡 Learning Outcomes remain the same:")
    print("  - Database schema design ✓")
    print("  - Data pipeline creation ✓")
    print("  - Workflow automation ✓")
    print("  - App development ✓")
print("="*70)
